# Exercise 04 — Logit Lens & Direct Logit Attribution (write it yourself)

Same task as `01_notebooks/04_logit_lens.ipynb`, core logic blanked. Self-checks
after each exercise. Solution: [`../01_notebooks/04_logit_lens.ipynb`](../01_notebooks/04_logit_lens.ipynb).


In [ ]:
import torch
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)

text = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(text)
logits, cache = model.run_with_cache(tokens)
correct_token = model.to_single_token(" Paris")


## Exercise 1 — logit lens

Compute `probs`: a 1D tensor where `probs[i]` is the model's predicted
probability of `" Paris"` at the last token position, if you stopped
accumulating the residual stream at snapshot `i`.

<details>
<summary>Hint</summary>

1. `cache.accumulated_resid(layer=-1, incl_mid=True, return_labels=True)`
   gives `(resid_stack, labels)` — a stack of residual-stream snapshots and
   their names.
2. `cache.apply_ln_to_stack(resid_stack, layer=-1)` applies the model's
   final LayerNorm correctly to the whole stack.
3. Multiply by `model.W_U` to get logits per snapshot, take the last
   position (`[:, 0, -1, :]`), softmax over the vocab dimension, and index
   `correct_token`.
</details>


In [ ]:
def logit_lens_probs(cache, model, correct_token):
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# self-check
probs = logit_lens_probs(cache, model, correct_token)
assert probs.shape == (2 * model.cfg.n_layers + 1,), f"expected {2 * model.cfg.n_layers + 1} points, got {probs.shape}"
assert probs[-1].item() > 5 * probs[0].item(), "confidence should build up substantially from layer 0 to the end"
assert probs[-1].item() > 0.01, f"final predicted probability seems too low: {probs[-1].item():.5f}"
print("Exercise 1 passed —", probs[-1].item())


## Exercise 2 — direct logit attribution (DLA) per head

Compute `dla`: a `[n_layers, n_heads]` tensor where `dla[l, h]` is how much
head `(l, h)` alone pushed the final logit for `" Paris"` up or down.

<details>
<summary>Hint</summary>

1. `cache.stack_head_results(layer=-1, return_labels=True)` gives you each
   head's contribution to the residual stream, stacked.
2. Apply the final LayerNorm the same way as Exercise 1.
3. `model.W_U[:, correct_token]` is the direction in residual-stream space
   that increases the `" Paris"` logit — dot each head's (LN-scaled)
   contribution at the last position against that direction, then reshape
   to `[n_layers, n_heads]`.
</details>


In [ ]:
def direct_logit_attribution(cache, model, correct_token):
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# self-check
dla = direct_logit_attribution(cache, model, correct_token)
assert dla.shape == (model.cfg.n_layers, model.cfg.n_heads)
assert dla.std().item() > 0, "all heads scored identically — something's wrong"
print("Exercise 2 passed — top head:", divmod(dla.flatten().argmax().item(), model.cfg.n_heads),
      "score", dla.flatten().max().item())


In [ ]:
import plotly.express as px

px.imshow(dla.cpu(), labels=dict(x="head", y="layer"), title="Direct logit attribution toward ' Paris'",
          color_continuous_scale="RdBu", color_continuous_midpoint=0).show()
